# Configuracion de RAG: extraccion, chunkeo tokenizado, carga en Qdrant y retrieval sobre el dataset de validacion

Notebook dividido en celdas para revisar cada paso por separado:

1. **Extraccion unificada** del PDF: texto narrativo continuo + tablas separadas y estructuradas + metadata de pagina.
2. **Chunkeo** con `RecursiveCharacterTextSplitter`, midiendo el tamano de cada chunk con el **tokenizer real de cada modelo de embedding**. Tamanos: 200, 500 y 800 tokens, todos con `chunk_overlap=0`.
3. **Carga en Qdrant**: una coleccion por combinacion (modelo de embedding, chunk_size), vectores densos, persistida en disco local. Incluye utilidades para ver el estado de las colecciones y borrarlas todas.
4. **Retrieval completo sobre el dataset de validacion** (`k=5`) para las 9 configuraciones, guardado en un unico JSON listo para usar despues con RAGAS.

**Variables de entorno necesarias** antes de correr las secciones 2, 3 y 4:
- `OPENROUTER_API_KEY` -> para `openai/text-embedding-3-large` via OpenRouter.
- `HF_TOKEN` -> para `google/embeddinggemma-300m` (modelo "gated": acepta la licencia en https://huggingface.co/google/embeddinggemma-300m con tu cuenta).


In [ ]:
%pip install -q pymupdf4llm langchain-text-splitters langchain-core tiktoken transformers sentence-transformers qdrant-client openai

## 1. Extraccion unificada del PDF

Una sola llamada a `pymupdf4llm.to_markdown(..., page_chunks=True)` nos da el markdown de cada pagina (tablas incluidas) y su numero de pagina. A partir de ahi:

- Sacamos las tablas de cada pagina con una regex de tablas Markdown, dejando un marcador de referencia en su lugar.
- Concatenamos el texto narrativo de todas las paginas nosotros mismos, con un espacio como separador (no un salto de parrafo), para no forzar un corte en el borde de pagina.
- Guardamos un mapa de offsets -> numero de pagina, para poder recuperar la(s) pagina(s) de cada chunk despues de trocear.


In [7]:
import re
import os
from typing import Dict, List, Optional, Tuple
from functools import lru_cache

import pymupdf4llm

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_core.documents import Document

# Ruta del PDF final a procesar (actualiza esto al documento completo)
pdf_path = "./archivo_fuente/7RAAC_Ingenierias Sistemas UV-24042025.pdf"

c:\Users\gomex\OneDrive\Escritorio\TG - Pruebas 2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Detecta tablas en formato Markdown (encabezado + fila separadora + filas de datos).
# Admite separadores con alineacion tipo :---, ---:, :---:
PATRON_TABLA_MD = re.compile(
    r'(^\|.*\|\s*\n'
    r'(?:\|[ \t]*:?-+:?[ \t]*\|.*\n)'
    r'(?:\|.*\|\s*\n?)+)',
    re.MULTILINE,
)


def _separar_tablas_de_texto(texto: str, numero_pagina: Optional[int]) -> Tuple[str, List[Dict]]:
    tablas_encontradas = PATRON_TABLA_MD.findall(texto)
    tablas = [
        {"numero_pagina": numero_pagina, "contenido_md": t.strip()}
        for t in tablas_encontradas
    ]

    contador = {"i": 0}

    def _reemplazo(match):
        contador["i"] += 1
        return f"\n[tabla {contador['i']} de la pagina {numero_pagina}]\n"

    texto_sin_tablas = PATRON_TABLA_MD.sub(_reemplazo, texto)
    return texto_sin_tablas, tablas

In [3]:
def extraer_documento_unificado(
    pdf_path: str,
    table_strategy: str = "lines_strict",
    header: bool = False,
    footer: bool = False,
) -> Dict:
    paginas = pymupdf4llm.to_markdown(
        pdf_path,
        page_chunks=True,
        table_strategy=table_strategy,
        header=header,
        footer=footer,
    )

    if isinstance(paginas, str):
        raise RuntimeError(
            "pymupdf4llm.to_markdown no devolvio una lista por pagina. "
            "Revisa tu version de pymupdf4llm / si tienes pymupdf-layout instalado."
        )

    partes_texto: List[str] = []
    mapa_paginas: List[Tuple[int, int, Optional[int]]] = []
    tablas_extraidas: List[Dict] = []
    offset_actual = 0

    for pagina in paginas:
        metadata_pagina = pagina.get("metadata", {})
        numero_pagina = metadata_pagina.get("page", metadata_pagina.get("page_number"))

        texto_pagina = pagina.get("text", "")
        texto_sin_tablas, tablas_pagina = _separar_tablas_de_texto(texto_pagina, numero_pagina)
        tablas_extraidas.extend(tablas_pagina)

        if partes_texto:
            partes_texto.append(" ")
            offset_actual += 1

        inicio = offset_actual
        partes_texto.append(texto_sin_tablas)
        offset_actual += len(texto_sin_tablas)
        fin = offset_actual

        mapa_paginas.append((inicio, fin, numero_pagina))

    for i, tabla in enumerate(tablas_extraidas):
        tabla["indice"] = i

    return {
        "texto_narrativo": "".join(partes_texto),
        "mapa_paginas": mapa_paginas,
        "tablas": tablas_extraidas,
    }


def _paginas_de_rango(mapa_paginas, inicio: int, fin: int) -> List[Optional[int]]:
    paginas = [
        num for (ini_p, fin_p, num) in mapa_paginas
        if ini_p < fin and fin_p > inicio and num is not None
    ]
    return paginas or [None]

In [4]:
extraido = extraer_documento_unificado(pdf_path)

documentos_tablas = [
    Document(
        page_content=t["contenido_md"],
        metadata={
            "tipo": "tabla",
            "pagina": t["numero_pagina"],
            "fuente": pdf_path,
            "indice_tabla": t["indice"],
        },
    )
    for t in extraido["tablas"]
]

print(f"Paginas procesadas: {len(extraido['mapa_paginas'])}")
print(f"Longitud del texto narrativo: {len(extraido['texto_narrativo'])} caracteres")
print(f"Tablas extraidas: {len(documentos_tablas)}")

Paginas procesadas: 330
Longitud del texto narrativo: 649953 caracteres
Tablas extraidas: 222


## 2. Chunkeo tokenizado por modelo de embedding

`chunk_size` se mide en **tokens del tokenizer real de cada modelo**:

- `openai/text-embedding-3-large` -> `cl100k_base` (tiktoken).
- `BAAI/bge-m3` -> tokenizer HuggingFace de `BAAI/bge-m3`.
- `google/embeddinggemma-300m` -> tokenizer HuggingFace de `google/embeddinggemma-300m`.

3 modelos x 3 chunk_sizes = 9 versiones de chunks de texto (las tablas no se re-chunkean, son las mismas para todos).


In [8]:
MODELOS_EMBEDDING = {
    "openai/text-embedding-3-large": {
        "proveedor": "openrouter",
        "dimension": 3072,
        "tokenizer_encoding": "cl100k_base",
    },
    "BAAI/bge-m3": {
        "proveedor": "sentence_transformers",
        "dimension": 1024,
        "tokenizer_id": "BAAI/bge-m3",
    },
    "google/embeddinggemma-300m": {
        "proveedor": "sentence_transformers",
        "dimension": 768,
        "tokenizer_id": "google/embeddinggemma-300m",
    },
}


@lru_cache(maxsize=None)
def _cargar_tiktoken(encoding_name: str):
    import tiktoken
    return tiktoken.get_encoding(encoding_name)


@lru_cache(maxsize=None)
def _cargar_tokenizer_hf(tokenizer_id: str):
    from transformers import AutoTokenizer
    return AutoTokenizer.from_pretrained(tokenizer_id, token=os.environ.get("HF_TOKEN"))


def obtener_contador_tokens(nombre_modelo: str):
    """Devuelve una funcion texto -> numero de tokens, usando el tokenizer real del modelo."""
    info = MODELOS_EMBEDDING[nombre_modelo]
    if info["proveedor"] == "openrouter":
        enc = _cargar_tiktoken(info["tokenizer_encoding"])
        return lambda texto: len(enc.encode(texto))
    tok = _cargar_tokenizer_hf(info["tokenizer_id"])
    return lambda texto: len(tok.encode(texto, add_special_tokens=False))

In [6]:
def construir_chunks_texto(texto_narrativo, mapa_paginas, chunk_size, chunk_overlap, contador_tokens, pdf_path, nombre_modelo):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=contador_tokens,
        add_start_index=True,
    )
    chunks = splitter.create_documents([texto_narrativo])

    documentos = []
    for chunk in chunks:
        inicio = chunk.metadata.get("start_index", 0)
        fin = inicio + len(chunk.page_content)
        paginas = _paginas_de_rango(mapa_paginas, inicio, fin)
        paginas_validas = [p for p in paginas if p is not None]

        documentos.append(Document(
            page_content=chunk.page_content,
            metadata={
                "tipo": "texto",
                "pagina_inicio": min(paginas_validas) if paginas_validas else None,
                "pagina_fin": max(paginas_validas) if paginas_validas else None,
                "fuente": pdf_path,
                "chunk_size_config": chunk_size,
                "modelo_embedding": nombre_modelo,
            },
        ))
    return documentos

In [7]:
CHUNK_SIZES = [300, 500, 800]
CHUNK_OVERLAP = 0

chunks_por_configuracion = {}

for nombre_modelo in MODELOS_EMBEDDING:
    contador_tokens = obtener_contador_tokens(nombre_modelo)
    for chunk_size in CHUNK_SIZES:
        docs = construir_chunks_texto(
            extraido["texto_narrativo"],
            extraido["mapa_paginas"],
            chunk_size,
            CHUNK_OVERLAP,
            contador_tokens,
            pdf_path,
            nombre_modelo,
        )
        chunks_por_configuracion[(nombre_modelo, chunk_size)] = docs
        largos = [contador_tokens(d.page_content) for d in docs]
        promedio = sum(largos) / len(largos) if largos else 0
        print(f"{nombre_modelo:32s} | chunk_size={chunk_size:4d} tokens -> {len(docs):4d} chunks (largo promedio real: {promedio:.0f} tokens)")

openai/text-embedding-3-large    | chunk_size= 300 tokens ->  751 chunks (largo promedio real: 224 tokens)
openai/text-embedding-3-large    | chunk_size= 500 tokens ->  416 chunks (largo promedio real: 406 tokens)
openai/text-embedding-3-large    | chunk_size= 800 tokens ->  245 chunks (largo promedio real: 690 tokens)
BAAI/bge-m3                      | chunk_size= 300 tokens ->  609 chunks (largo promedio real: 236 tokens)
BAAI/bge-m3                      | chunk_size= 500 tokens ->  338 chunks (largo promedio real: 426 tokens)
BAAI/bge-m3                      | chunk_size= 800 tokens ->  200 chunks (largo promedio real: 719 tokens)
google/embeddinggemma-300m       | chunk_size= 300 tokens ->  622 chunks (largo promedio real: 236 tokens)
google/embeddinggemma-300m       | chunk_size= 500 tokens ->  350 chunks (largo promedio real: 422 tokens)
google/embeddinggemma-300m       | chunk_size= 800 tokens ->  208 chunks (largo promedio real: 711 tokens)


## 3. Vectores densos en Qdrant (persistente, por modelo y chunk_size)

Una coleccion por combinacion `(modelo, chunk_size)`, configurada solo con vector denso y persistida en disco local (`QdrantClient(path=...)`, sin servidor).

Nota: el modo `path=` no admite acceso concurrente desde dos procesos/kernels a la vez; si reinicias el kernel, cierra bien el cliente antes de volver a abrirlo.

Las tablas no dependen de `chunk_size`, asi que se embeben **una sola vez por modelo** y se reutilizan en las 3 colecciones de ese modelo.


In [11]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
import uuid

RUTA_QDRANT = "./qdrant_data"  # persistencia local en disco, sin servidor

# Cierra el cliente anterior si ya existe en el kernel (evita el lock al re-ejecutar la celda)
try:
    client_qdrant.close()
except NameError:
    pass
except Exception as e:
    print(f"Aviso: no se pudo cerrar el cliente anterior ({e})")

client_qdrant = QdrantClient(path=RUTA_QDRANT)


def _slug(texto: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", texto.lower()).strip("_")


def nombre_coleccion(nombre_modelo: str, chunk_size: int) -> str:
    return f"rag_{_slug(nombre_modelo)}_cs{chunk_size}"


def asegurar_coleccion(client, nombre: str, dimension: int):
    if not client.collection_exists(nombre):
        client.create_collection(
            collection_name=nombre,
            vectors_config=VectorParams(size=dimension, distance=Distance.COSINE),
        )

In [13]:
from openai import OpenAI

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
_cliente_openrouter = None


def _obtener_cliente_openrouter():
    global _cliente_openrouter
    if _cliente_openrouter is None:
        if not OPENROUTER_API_KEY:
            raise RuntimeError("Define la variable de entorno OPENROUTER_API_KEY")
        _cliente_openrouter = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
    return _cliente_openrouter


def embeber_openrouter(textos, nombre_modelo, tam_lote=96):
    if not textos:
        return []
    cliente = _obtener_cliente_openrouter()
    vectores = []
    for i in range(0, len(textos), tam_lote):
        lote = textos[i:i + tam_lote]
        resp = cliente.embeddings.create(model=nombre_modelo, input=lote)
        vectores.extend([item.embedding for item in resp.data])
    return vectores


_modelos_st_cache = {}


def _obtener_modelo_st(nombre_modelo):
    if nombre_modelo not in _modelos_st_cache:
        from sentence_transformers import SentenceTransformer
        _modelos_st_cache[nombre_modelo] = SentenceTransformer(nombre_modelo, token=os.environ.get("HF_TOKEN"))
    return _modelos_st_cache[nombre_modelo]


def embeber_sentence_transformers(textos, nombre_modelo, tam_lote=32):
    if not textos:
        return []
    modelo = _obtener_modelo_st(nombre_modelo)
    if hasattr(modelo, "encode_document"):
        vectores = modelo.encode_document(textos, batch_size=tam_lote)
    else:
        vectores = modelo.encode(textos, batch_size=tam_lote, normalize_embeddings=True)
    return [v.tolist() for v in vectores]


def embeber_textos(textos, nombre_modelo):
    info = MODELOS_EMBEDDING[nombre_modelo]
    if info["proveedor"] == "openrouter":
        return embeber_openrouter(textos, nombre_modelo)
    return embeber_sentence_transformers(textos, nombre_modelo)

In [10]:
for nombre_modelo, info in MODELOS_EMBEDDING.items():
    print(f"\n=== Modelo: {nombre_modelo} ===")
    dimension = info["dimension"]

    # Las tablas no dependen de chunk_size: se embeben una sola vez por modelo
    textos_tablas = [d.page_content for d in documentos_tablas]
    vectores_tablas = embeber_textos(textos_tablas, nombre_modelo)

    for chunk_size in CHUNK_SIZES:
        docs_texto = chunks_por_configuracion[(nombre_modelo, chunk_size)]
        textos_chunks = [d.page_content for d in docs_texto]
        vectores_chunks = embeber_textos(textos_chunks, nombre_modelo)

        coleccion = nombre_coleccion(nombre_modelo, chunk_size)
        asegurar_coleccion(client_qdrant, coleccion, dimension)

        puntos = [
            PointStruct(id=str(uuid.uuid4()), vector=vector, payload={"page_content": doc.page_content, **doc.metadata})
            for doc, vector in zip(docs_texto, vectores_chunks)
        ]
        puntos += [
            PointStruct(id=str(uuid.uuid4()), vector=vector, payload={"page_content": doc.page_content, **doc.metadata, "chunk_size_config": chunk_size})
            for doc, vector in zip(documentos_tablas, vectores_tablas)
        ]

        if puntos:
            client_qdrant.upsert(collection_name=coleccion, points=puntos)

        print(f"  chunk_size={chunk_size:4d} -> coleccion \'{coleccion}\' con {len(puntos)} puntos ({len(docs_texto)} texto + {len(documentos_tablas)} tablas)")


=== Modelo: openai/text-embedding-3-large ===
  chunk_size= 300 -> coleccion 'rag_openai_text_embedding_3_large_cs300' con 973 puntos (751 texto + 222 tablas)
  chunk_size= 500 -> coleccion 'rag_openai_text_embedding_3_large_cs500' con 638 puntos (416 texto + 222 tablas)
  chunk_size= 800 -> coleccion 'rag_openai_text_embedding_3_large_cs800' con 467 puntos (245 texto + 222 tablas)

=== Modelo: BAAI/bge-m3 ===


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 22442.32it/s]


  chunk_size= 300 -> coleccion 'rag_baai_bge_m3_cs300' con 831 puntos (609 texto + 222 tablas)
  chunk_size= 500 -> coleccion 'rag_baai_bge_m3_cs500' con 560 puntos (338 texto + 222 tablas)
  chunk_size= 800 -> coleccion 'rag_baai_bge_m3_cs800' con 422 puntos (200 texto + 222 tablas)

=== Modelo: google/embeddinggemma-300m ===


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 886.60it/s]


  chunk_size= 300 -> coleccion 'rag_google_embeddinggemma_300m_cs300' con 844 puntos (622 texto + 222 tablas)
  chunk_size= 500 -> coleccion 'rag_google_embeddinggemma_300m_cs500' con 572 puntos (350 texto + 222 tablas)
  chunk_size= 800 -> coleccion 'rag_google_embeddinggemma_300m_cs800' con 430 puntos (208 texto + 222 tablas)


### Utilidades: ver estado y borrar colecciones

Util cuando cambias de PDF de prueba a PDF final y quieres partir de cero, o simplemente para confirmar que una coleccion cambio (numero de puntos y tamano en disco).


In [11]:
from pathlib import Path


def tamano_carpeta_mb(ruta) -> float:
    total = sum(f.stat().st_size for f in Path(ruta).rglob("*") if f.is_file())
    return total / (1024 * 1024)


def resumen_qdrant(client, ruta_datos):
    colecciones = client.get_collections().collections
    if not colecciones:
        print("No hay colecciones.")
    for c in colecciones:
        info = client.get_collection(c.name)
        print(f"  {c.name:40s} -> {info.points_count} puntos | status={info.status}")
    print(f"Tamano total de \'{ruta_datos}\': {tamano_carpeta_mb(ruta_datos):.2f} MB")


print("Estado actual de Qdrant:")
resumen_qdrant(client_qdrant, RUTA_QDRANT)

Estado actual de Qdrant:
  rag_openai_text_embedding_3_large_cs300  -> 973 puntos | status=green
  rag_openai_text_embedding_3_large_cs500  -> 638 puntos | status=green
  rag_openai_text_embedding_3_large_cs800  -> 467 puntos | status=green
  rag_baai_bge_m3_cs300                    -> 831 puntos | status=green
  rag_baai_bge_m3_cs500                    -> 560 puntos | status=green
  rag_baai_bge_m3_cs800                    -> 422 puntos | status=green
  rag_google_embeddinggemma_300m_cs300     -> 844 puntos | status=green
  rag_google_embeddinggemma_300m_cs500     -> 572 puntos | status=green
  rag_google_embeddinggemma_300m_cs800     -> 430 puntos | status=green
Tamano total de './qdrant_data': 97.13 MB


In [39]:
# Cambia a True SOLO cuando quieras borrar TODAS las colecciones existentes
# (por ejemplo, al pasar del PDF de prueba al PDF final, para no mezclar datos).
CONFIRMAR_BORRADO = False

if CONFIRMAR_BORRADO:
    for c in client_qdrant.get_collections().collections:
        client_qdrant.delete_collection(c.name)
        print(f"Borrada: {c.name}")
    print("\nEstado despues de borrar:")
    resumen_qdrant(client_qdrant, RUTA_QDRANT)
else:
    print("CONFIRMAR_BORRADO=False, no se borro nada.")

CONFIRMAR_BORRADO=False, no se borro nada.


### Ejemplo rapido: preguntar directamente a una coleccion

Sanity check antes de correr todo el dataset de validacion.


In [14]:
def embeber_consulta(pregunta: str, nombre_modelo: str):
    """Igual que embeber_textos, pero para el lado de la consulta (no del documento)."""
    info = MODELOS_EMBEDDING[nombre_modelo]
    if info["proveedor"] == "openrouter":
        cliente = _obtener_cliente_openrouter()
        resp = cliente.embeddings.create(model=nombre_modelo, input=[pregunta])
        return resp.data[0].embedding
    modelo = _obtener_modelo_st(nombre_modelo)
    if hasattr(modelo, "encode_query"):
        # embeddinggemma: prompt distinto para consultas que para documentos
        return modelo.encode_query(pregunta).tolist()
    return modelo.encode(pregunta, normalize_embeddings=True).tolist()


# --- ejemplo: preguntar directamente a UNA coleccion ---
nombre_modelo = "BAAI/bge-m3"     # cambia el modelo aqui
chunk_size = 500                  # y el chunk_size aqui
pregunta = "¿cuál es el objetivo del programa de ingenieria en sistemas?"   # tu pregunta

coleccion = nombre_coleccion(nombre_modelo, chunk_size)
vector_pregunta = embeber_consulta(pregunta, nombre_modelo)

resultados = client_qdrant.query_points(
    collection_name=coleccion,
    query=vector_pregunta,
    limit=5,
    with_payload=True,
)

for punto in resultados.points:
    pagina = punto.payload.get("pagina_inicio", punto.payload.get("pagina"))
    print(f"score={punto.score:.4f} | tipo={punto.payload.get('tipo')} | pagina={pagina}")
    print(punto.payload.get("page_content")[:300])
    print("---")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 21799.74it/s]
c:\Users\gomex\OneDrive\Escritorio\TG - Pruebas 2\venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gomex\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


score=0.7051 | tipo=texto | pagina=39
De acuerdo con la Misión y la Visión de la Universidad del Valle, el programa académico de Ingeniería de Sistemas tiene como **objetivos específicos:** 

- Formar estudiantes con capacidad para interpretar y aplicar las ciencias pertinentes a su especialidad, así como los fundamentos, los métodos y 
---
score=0.6770 | tipo=texto | pagina=38
En línea con lo establecido en el PEP, el programa tiene como objetivos formar profesionales íntegros y responsables, promover la interdisciplinariedad mediante una sólida formación básica en áreas diversas, y desarrollar habilidades de pensamiento crítico, creativo y sistémico. Además, se enfoca en
---
score=0.6728 | tipo=texto | pagina=35
- _Los objetivos del Programa Académico de Ingeniería de Sistemas se alinean con la Misión y la Visión de la Universidad del Valle y su logro es orientado a través del Proyecto Educativo del Programa Académico (PEP) de Ingeniería de Sistemas._ ○ _Continúa así y el Programa de

## 4. Retrieval completo sobre el dataset de validacion (las 9 configuraciones)

Corre el retrieval (`k=5`) de **todas** las preguntas del dataset de validacion contra las **9 colecciones** (3 modelos x 3 chunk_sizes), y guarda todo en un unico JSON.

Cada pregunta se embebe **una sola vez por modelo** (no una vez por chunk_size) y esa consulta se reutiliza contra las 3 colecciones de ese modelo, para no gastar llamadas de mas.

El JSON de salida queda organizado por configuracion, con `pregunta`, `respuesta_esperada` y `evidencia_gold` (tu gold) junto a `contextos_recuperados` (los k=5 textos que trajo el retrieval) — listos para mapear despues a `question` / `ground_truth` / `contexts` de RAGAS. Eso lo dejamos para la siguiente celda/sesion.


In [3]:
import json
from pathlib import Path

RUTA_DATASET = "./validacion/dataset_validacion_final.json"
RUTA_SALIDA = "./validacion/resultados_retrieval_2.json"
K = 5

with open(RUTA_DATASET, "r", encoding="utf-8") as f:
    dataset = json.load(f)

preguntas = dataset["pares"]
print(f"Preguntas cargadas: {len(preguntas)}")

Preguntas cargadas: 296


In [14]:
def embeber_consultas_batch(textos, nombre_modelo, tam_lote=64):
    """Version en lote de embeber_consulta: una sola pasada por modelo en vez de una por pregunta."""
    if not textos:
        return []
    info = MODELOS_EMBEDDING[nombre_modelo]

    if info["proveedor"] == "openrouter":
        cliente = _obtener_cliente_openrouter()
        vectores = []
        for i in range(0, len(textos), tam_lote):
            lote = textos[i:i + tam_lote]
            resp = cliente.embeddings.create(model=nombre_modelo, input=lote)
            vectores.extend([item.embedding for item in resp.data])
        return vectores

    modelo = _obtener_modelo_st(nombre_modelo)
    try:
        if hasattr(modelo, "encode_query"):
            vectores = modelo.encode_query(textos, batch_size=tam_lote)
        else:
            vectores = modelo.encode(textos, batch_size=tam_lote, normalize_embeddings=True)
    except Exception:
        # fallback por si encode_query no admite lote en tu version instalada
        metodo = modelo.encode_query if hasattr(modelo, "encode_query") else modelo.encode
        vectores = [metodo(t) for t in textos]
    return [v.tolist() for v in vectores]

In [15]:
textos_preguntas = [p["pregunta"] for p in preguntas]
configuraciones_resultado = []

for nombre_modelo in MODELOS_EMBEDDING:
    print(f"\n=== Embebiendo {len(textos_preguntas)} preguntas con {nombre_modelo} ===")
    vectores_preguntas = embeber_consultas_batch(textos_preguntas, nombre_modelo)

    for chunk_size in CHUNK_SIZES:
        coleccion = nombre_coleccion(nombre_modelo, chunk_size)
        print(f"  Retrieval en \'{coleccion}\' (k={K})...")

        resultados_config = []
        for pregunta_info, vector in zip(preguntas, vectores_preguntas):
            hits = client_qdrant.query_points(
                collection_name=coleccion,
                query=vector,
                limit=K,
                with_payload=True,
            ).points

            resultados_config.append({
                "id": pregunta_info["id"],
                "pregunta": pregunta_info["pregunta"],
                "respuesta_esperada": pregunta_info.get("respuesta_esperada"),
                "evidencia_gold": pregunta_info.get("evidencia_gold"),
                "tipo": pregunta_info.get("tipo"),
                "dificultad": pregunta_info.get("dificultad"),
                "tipo_evidencia": pregunta_info.get("tipo_evidencia"),
                "seccion": pregunta_info.get("seccion"),
                "contextos_recuperados": [h.payload.get("page_content") for h in hits],
                "metadatos_recuperados": [
                    {
                        "score": h.score,
                        "tipo": h.payload.get("tipo"),
                        "pagina_inicio": h.payload.get("pagina_inicio", h.payload.get("pagina")),
                        "pagina_fin": h.payload.get("pagina_fin"),
                    }
                    for h in hits
                ],
            })

        configuraciones_resultado.append({
            "modelo_embedding": nombre_modelo,
            "chunk_size": chunk_size,
            "coleccion": coleccion,
            "k": K,
            "resultados": resultados_config,
        })

salida = {
    "k": K,
    "dataset_fuente": dataset.get("documento"),
    "total_preguntas": len(preguntas),
    "configuraciones": configuraciones_resultado,
}

Path(RUTA_SALIDA).parent.mkdir(parents=True, exist_ok=True)
with open(RUTA_SALIDA, "w", encoding="utf-8") as f:
    json.dump(salida, f, ensure_ascii=False, indent=2)

print(f"\nListo. {len(configuraciones_resultado)} configuraciones guardadas en {RUTA_SALIDA}")


=== Embebiendo 296 preguntas con openai/text-embedding-3-large ===
  Retrieval en 'rag_openai_text_embedding_3_large_cs300' (k=5)...
  Retrieval en 'rag_openai_text_embedding_3_large_cs500' (k=5)...
  Retrieval en 'rag_openai_text_embedding_3_large_cs800' (k=5)...

=== Embebiendo 296 preguntas con BAAI/bge-m3 ===
  Retrieval en 'rag_baai_bge_m3_cs300' (k=5)...
  Retrieval en 'rag_baai_bge_m3_cs500' (k=5)...
  Retrieval en 'rag_baai_bge_m3_cs800' (k=5)...

=== Embebiendo 296 preguntas con google/embeddinggemma-300m ===
  Retrieval en 'rag_google_embeddinggemma_300m_cs300' (k=5)...
  Retrieval en 'rag_google_embeddinggemma_300m_cs500' (k=5)...
  Retrieval en 'rag_google_embeddinggemma_300m_cs800' (k=5)...

Listo. 9 configuraciones guardadas en ./validacion/resultados_retrieval_2.json


In [16]:
# Sanity check rapido del archivo generado
tamano_mb = Path(RUTA_SALIDA).stat().st_size / (1024 * 1024)
print(f"Archivo: {RUTA_SALIDA} ({tamano_mb:.2f} MB)")
for c in configuraciones_resultado:
    n_con_contexto = sum(1 for r in c["resultados"] if r["contextos_recuperados"])
    print(f"  {c['modelo_embedding']:32s} | cs={c['chunk_size']:4d} -> {len(c['resultados'])} preguntas, {n_con_contexto} con al menos 1 contexto recuperado")

Archivo: ./validacion/resultados_retrieval_2.json (27.64 MB)
  openai/text-embedding-3-large    | cs= 300 -> 296 preguntas, 296 con al menos 1 contexto recuperado
  openai/text-embedding-3-large    | cs= 500 -> 296 preguntas, 296 con al menos 1 contexto recuperado
  openai/text-embedding-3-large    | cs= 800 -> 296 preguntas, 296 con al menos 1 contexto recuperado
  BAAI/bge-m3                      | cs= 300 -> 296 preguntas, 296 con al menos 1 contexto recuperado
  BAAI/bge-m3                      | cs= 500 -> 296 preguntas, 296 con al menos 1 contexto recuperado
  BAAI/bge-m3                      | cs= 800 -> 296 preguntas, 296 con al menos 1 contexto recuperado
  google/embeddinggemma-300m       | cs= 300 -> 296 preguntas, 296 con al menos 1 contexto recuperado
  google/embeddinggemma-300m       | cs= 500 -> 296 preguntas, 296 con al menos 1 contexto recuperado
  google/embeddinggemma-300m       | cs= 800 -> 296 preguntas, 296 con al menos 1 contexto recuperado


In [17]:
# --- Experimento: Gemma cs800 con k reducido ---
# Hipotesis: a cs800 el recall de Gemma ya es alto (0.747) pero la precision
# cae respecto a cs500 (0.665 vs 0.697) -> el chunk extra probablemente solo
# agrega ruido. Si el chunk correcto ya suele estar en las primeras
# posiciones, bajar k deberia recuperar precision sin perder mucho recall.
NOMBRE_MODELO_EXP = "google/embeddinggemma-300m"
CHUNK_SIZE_EXP = 800
K_EXP = 3
RUTA_SALIDA_EXP = f"./validacion/resultados_retrieval_gemma_cs{CHUNK_SIZE_EXP}_k{K_EXP}.json"

coleccion_exp = nombre_coleccion(NOMBRE_MODELO_EXP, CHUNK_SIZE_EXP)
print(f"Experimento: {NOMBRE_MODELO_EXP} | chunk_size={CHUNK_SIZE_EXP} | k={K_EXP} -> coleccion '{coleccion_exp}'")

vectores_preguntas_exp = embeber_consultas_batch(textos_preguntas, NOMBRE_MODELO_EXP)

resultados_exp = []
for pregunta_info, vector in zip(preguntas, vectores_preguntas_exp):
    hits = client_qdrant.query_points(
        collection_name=coleccion_exp,
        query=vector,
        limit=K_EXP,
        with_payload=True,
    ).points

    resultados_exp.append({
        "id": pregunta_info["id"],
        "pregunta": pregunta_info["pregunta"],
        "respuesta_esperada": pregunta_info.get("respuesta_esperada"),
        "evidencia_gold": pregunta_info.get("evidencia_gold"),
        "tipo": pregunta_info.get("tipo"),
        "dificultad": pregunta_info.get("dificultad"),
        "tipo_evidencia": pregunta_info.get("tipo_evidencia"),
        "seccion": pregunta_info.get("seccion"),
        "contextos_recuperados": [h.payload.get("page_content") for h in hits],
        "metadatos_recuperados": [
            {
                "score": h.score,
                "tipo": h.payload.get("tipo"),
                "pagina_inicio": h.payload.get("pagina_inicio", h.payload.get("pagina")),
                "pagina_fin": h.payload.get("pagina_fin"),
            }
            for h in hits
        ],
    })

salida_exp = {
    "modelo_embedding": NOMBRE_MODELO_EXP,
    "chunk_size": CHUNK_SIZE_EXP,
    "coleccion": coleccion_exp,
    "k": K_EXP,
    "experimento": f"k reducido a {K_EXP} en cs{CHUNK_SIZE_EXP} (hipotesis: menos ruido, mejor precision, sin perder mucho recall)",
    "dataset_fuente": dataset.get("documento"),
    "total_preguntas": len(preguntas),
    "resultados": resultados_exp,
}

Path(RUTA_SALIDA_EXP).parent.mkdir(parents=True, exist_ok=True)
with open(RUTA_SALIDA_EXP, "w", encoding="utf-8") as f:
    json.dump(salida_exp, f, ensure_ascii=False, indent=2)

tamano_mb = Path(RUTA_SALIDA_EXP).stat().st_size / (1024 * 1024)
n_con_contexto = sum(1 for r in resultados_exp if r["contextos_recuperados"])
print(f"\nListo. Guardado en {RUTA_SALIDA_EXP} ({tamano_mb:.2f} MB)")
print(f"{len(resultados_exp)} preguntas, {n_con_contexto} con al menos 1 contexto recuperado")

Experimento: google/embeddinggemma-300m | chunk_size=800 | k=3 -> coleccion 'rag_google_embeddinggemma_300m_cs800'

Listo. Guardado en ./validacion/resultados_retrieval_gemma_cs800_k3.json (2.61 MB)
296 preguntas, 296 con al menos 1 contexto recuperado


In [ ]:
##Versión donde se mide la latencia
# --- Medicion de latencia: las 9 configuraciones base (denso puro) ---
# Mismas colecciones y misma embeber_consulta de arriba. Nada se re-indexa.
import time
import numpy as np

CHUNK_SIZES = [300, 500, 800]   # igual que en la seccion 2 (asi no hay que correr el chunkeo)
N_CALENTAMIENTO = 5
RUTA_SALIDA_LAT = "./validacion/resultados_retrieval_2_latencia.json"

textos_preguntas = [p["pregunta"] for p in preguntas]
modelos = list(MODELOS_EMBEDDING)


def _resumen(valores):
    a = np.asarray(valores, dtype=float)
    return {
        "media_ms": float(a.mean()),
        "mediana_ms": float(np.median(a)),
        "p95_ms": float(np.percentile(a, 95)),
        "min_ms": float(a.min()),
        "max_ms": float(a.max()),
        "desv_std_ms": float(a.std(ddof=1)),
    }


# 1) Calentamiento (no se mide): carga perezosa de pesos, conexion HTTP, primeras lecturas de Qdrant
for nombre_modelo in modelos:
    for texto in textos_preguntas[:N_CALENTAMIENTO]:
        v = embeber_consulta(texto, nombre_modelo)
        for cs in CHUNK_SIZES:
            client_qdrant.query_points(collection_name=nombre_coleccion(nombre_modelo, cs), query=v, limit=K, with_payload=True)

# 2) Medicion: una consulta a la vez, modelos intercalados y con orden rotado por pregunta
lat = {(m, cs): {"embedding_consulta": [], "busqueda_densa": [], "total": []} for m in modelos for cs in CHUNK_SIZES}
contextos = {(m, cs): [] for m in modelos for cs in CHUNK_SIZES}

for i, texto in enumerate(textos_preguntas):
    r = i % len(modelos)
    for nombre_modelo in modelos[r:] + modelos[:r]:
        t0 = time.perf_counter()
        vector = embeber_consulta(texto, nombre_modelo)
        lat_emb = (time.perf_counter() - t0) * 1000

        for cs in CHUNK_SIZES:
            t0 = time.perf_counter()
            hits = client_qdrant.query_points(
                collection_name=nombre_coleccion(nombre_modelo, cs),
                query=vector,
                limit=K,
                with_payload=True,
            ).points
            lat_bus = (time.perf_counter() - t0) * 1000

            d = lat[(nombre_modelo, cs)]
            d["embedding_consulta"].append(lat_emb)
            d["busqueda_densa"].append(lat_bus)
            d["total"].append(lat_emb + lat_bus)
            contextos[(nombre_modelo, cs)].append([h.payload.get("page_content") for h in hits])

    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(textos_preguntas)} preguntas")

# 3) Verificacion: lo cronometrado debe ser el mismo retrieval que ya evaluaste con RAGAS
with open(RUTA_SALIDA, "r", encoding="utf-8") as f:
    evaluado = {(c["modelo_embedding"], c["chunk_size"]): c["resultados"] for c in json.load(f)["configuraciones"]}

configuraciones_lat = []
print(f"\n{'modelo':32s} {'cs':>4s} | {'emb med':>8s} {'busq med':>8s} {'total med':>9s} {'total p95':>9s} | coincide con evaluado")
for m in modelos:
    for cs in CHUNK_SIZES:
        previos = [r["contextos_recuperados"] for r in evaluado[(m, cs)]]
        iguales = sum(a == b for a, b in zip(contextos[(m, cs)], previos))
        res = {etapa: _resumen(v) for etapa, v in lat[(m, cs)].items()}
        print(f"{m:32s} {cs:4d} | {res['embedding_consulta']['mediana_ms']:8.1f} {res['busqueda_densa']['mediana_ms']:8.1f} "
              f"{res['total']['mediana_ms']:9.1f} {res['total']['p95_ms']:9.1f} | {iguales}/{len(previos)}")

        configuraciones_lat.append({
            "modelo_embedding": m,
            "chunk_size": cs,
            "coleccion": nombre_coleccion(m, cs),
            "k": K,
            "resumen_latencias_ms": res,
            "contextos_identicos_a_evaluacion": iguales,
            "latencias_por_pregunta_ms": [
                {"id": p["id"], "embedding_consulta": e, "busqueda_densa": b, "total": t}
                for p, e, b, t in zip(preguntas, *lat[(m, cs)].values())
            ],
        })

salida_lat = {
    "k": K,
    "dataset_fuente": dataset.get("documento"),
    "total_preguntas": len(preguntas),
    "metodo": "una consulta a la vez; calentamiento de "
              f"{N_CALENTAMIENTO} consultas por modelo; modelos intercalados con orden rotado; "
              "embedding de consulta medido una vez por modelo y compartido por sus 3 chunk_sizes",
    "configuraciones": configuraciones_lat,
}



Path(RUTA_SALIDA_LAT).parent.mkdir(parents=True, exist_ok=True)
with open(RUTA_SALIDA_LAT, "w", encoding="utf-8") as f:
    json.dump(salida_lat, f, ensure_ascii=False, indent=2)
print(f"\nGuardado en {RUTA_SALIDA_LAT}")

Loading weights: 100%|██████████| 314/314 [00:03<00:00, 81.10it/s]
c:\Users\gomex\OneDrive\Escritorio\TG - Pruebas 2\venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gomex\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


  50/296 preguntas
  100/296 preguntas
  150/296 preguntas
  200/296 preguntas
  250/296 preguntas

modelo                             cs |  emb med busq med total med total p95 | coincide con evaluado
openai/text-embedding-3-large     300 |    614.2     29.7     642.3     854.2 | 282/296
openai/text-embedding-3-large     500 |    614.2     20.6     633.5     843.8 | 287/296
openai/text-embedding-3-large     800 |    614.2     15.8     629.4     839.5 | 284/296
BAAI/bge-m3                       300 |    263.2      6.1     268.5     348.4 | 296/296
BAAI/bge-m3                       500 |    263.2      3.8     267.2     344.1 | 296/296
BAAI/bge-m3                       800 |    263.2      3.6     266.4     345.7 | 296/296
google/embeddinggemma-300m        300 |    110.0      4.1     113.6     179.7 | 296/296
google/embeddinggemma-300m        500 |    110.0      3.1     112.9     177.6 | 296/296
google/embeddinggemma-300m        800 |    110.0      2.1     112.3     174.5 | 296/296

Guard

## Notas finales

- Las 9 colecciones quedan persistidas en `RUTA_QDRANT` (`./qdrant_data`). Para volver a consultarlas despues sin recorrer todo el notebook, basta con `QdrantClient(path=RUTA_QDRANT)`.
- `resultados_retrieval.json` (en `RUTA_SALIDA`) queda organizado por configuracion (`modelo_embedding` + `chunk_size`), cada una con sus 296 preguntas y los `k=5` contextos recuperados. Es el insumo para, en la siguiente etapa, generar respuestas y evaluar con RAGAS.
- Si vuelves a correr la celda de indexado (seccion 3), los `PointStruct` se crean con un `id` nuevo (uuid4) cada vez, asi que se duplican los puntos en vez de sobreescribirse. Usa `CONFIRMAR_BORRADO = True` antes de re-indexar si quieres partir de cero.
- Recuerda: con `embeddinggemma` las consultas se embeben con `encode_query(...)`, no con `encode_document(...)` — ya esta resuelto en `embeber_consulta` / `embeber_consultas_batch`.
